# AI4DSNI Analysis Notebook

This notebook provides utilities for:
- Exploring the dataset
- Evaluating trained models
- Visualizing results

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path().absolute().parent
sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch

from src.data import SequenceEncoder, DSNIDataset, load_fasta, get_splits
from src.models import create_encoder, create_decoder
from src.train import DSNIModule

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

## 1. Data Exploration

In [ ]:
# Load data (replace with actual paths)
DATA_DIR = PROJECT_ROOT / "data"
FASTA_PATH = DATA_DIR / "sequences.fasta"
METADATA_PATH = DATA_DIR / "metadata.csv"

# Check if data exists
if FASTA_PATH.exists() and METADATA_PATH.exists():
    ids, sequences = load_fasta(FASTA_PATH)
    metadata = pd.read_csv(METADATA_PATH)
    print(f"Loaded {len(sequences)} sequences")
    print(f"Metadata columns: {metadata.columns.tolist()}")
else:
    print("Data files not found. Creating dummy data for demonstration.")
    from src.train import _create_dummy_data
    sequences, metadata = _create_dummy_data(n_samples=200)
    print(f"Created {len(sequences)} dummy sequences")

In [ ]:
# Display metadata summary
metadata.head()

In [ ]:
# Sequence length distribution
seq_lengths = [len(s) for s in sequences]

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(seq_lengths, bins=50, edgecolor='white', alpha=0.7)
ax.axvline(np.mean(seq_lengths), color='red', linestyle='--', label=f'Mean: {np.mean(seq_lengths):.0f}')
ax.axvline(np.median(seq_lengths), color='green', linestyle='--', label=f'Median: {np.median(seq_lengths):.0f}')
ax.set_xlabel('Sequence Length')
ax.set_ylabel('Count')
ax.set_title('Distribution of Sequence Lengths')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Label distributions
tasks = ['temperature', 'ph', 'oxygen', 'media']
available_tasks = [t for t in tasks if t in metadata.columns]

if available_tasks:
    fig, axes = plt.subplots(1, len(available_tasks), figsize=(4*len(available_tasks), 4))
    if len(available_tasks) == 1:
        axes = [axes]
    
    for ax, task in zip(axes, available_tasks):
        counts = metadata[task].value_counts()
        ax.bar(counts.index, counts.values, edgecolor='white')
        ax.set_title(f'{task.capitalize()} Distribution')
        ax.set_xlabel(task)
        ax.set_ylabel('Count')
        plt.setp(ax.get_xticklabels(), rotation=45, ha='right')
    
    plt.tight_layout()
    plt.show()

## 2. Model Loading and Evaluation

In [ ]:
# Load a trained model checkpoint
CHECKPOINT_DIR = PROJECT_ROOT / "outputs" / "checkpoints"

def load_model(checkpoint_path: Path) -> DSNIModule:
    """Load a trained model from checkpoint."""
    return DSNIModule.load_from_checkpoint(checkpoint_path)

# List available checkpoints
if CHECKPOINT_DIR.exists():
    checkpoints = list(CHECKPOINT_DIR.glob("*.ckpt"))
    print(f"Found {len(checkpoints)} checkpoints:")
    for ckpt in checkpoints[:5]:
        print(f"  - {ckpt.name}")
else:
    print("No checkpoints found. Train a model first.")

In [ ]:
# Create a model for demonstration
encoder = create_encoder(
    encoder_type="flat",
    config={
        "embedding_dim": 128,
        "hidden_dim": 256,
        "flat": {"num_conv_layers": 4},
    }
)

decoder = create_decoder(
    input_dim=encoder.output_dim,
    config={
        "decoder": {
            "hidden_dims": [128, 64],
            "tasks": {
                "temperature": {"num_classes": 3},
                "ph": {"num_classes": 3},
                "oxygen": {"num_classes": 2},
                "media": {"num_classes": 4},
            }
        }
    }
)

print(f"Encoder output dim: {encoder.output_dim}")
print(f"Decoder tasks: {list(decoder.heads.keys())}")

## 3. Prediction and Visualization

In [ ]:
# Helper function for predictions
def predict(model: DSNIModule, sequences: list, max_len: int = 1500):
    """Make predictions on sequences."""
    model.eval()
    encoder = SequenceEncoder(max_len=max_len)
    
    results = []
    with torch.no_grad():
        for seq in sequences:
            encoded = encoder(seq).unsqueeze(0)
            mask = (encoded != 0).long()
            
            logits = model(encoded, mask)
            
            preds = {}
            probs = {}
            for task, task_logits in logits.items():
                task_probs = torch.softmax(task_logits, dim=-1)
                preds[task] = task_logits.argmax(dim=-1).item()
                probs[task] = task_probs.squeeze().tolist()
            
            results.append({"predictions": preds, "probabilities": probs})
    
    return results

In [ ]:
# Confusion matrix plotting
from sklearn.metrics import confusion_matrix

def plot_confusion_matrix(y_true, y_pred, labels, title="Confusion Matrix"):
    """Plot a confusion matrix."""
    cm = confusion_matrix(y_true, y_pred)
    
    fig, ax = plt.subplots(figsize=(6, 5))
    sns.heatmap(
        cm, 
        annot=True, 
        fmt='d', 
        cmap='Blues',
        xticklabels=labels,
        yticklabels=labels,
        ax=ax
    )
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)
    plt.tight_layout()
    return fig

## 4. Embedding Visualization

In [ ]:
from sklearn.manifold import TSNE

def visualize_embeddings(
    embeddings: np.ndarray,
    labels: np.ndarray,
    label_names: list,
    title: str = "Embedding Visualization"
):
    """Visualize embeddings using t-SNE."""
    # Reduce dimensionality
    tsne = TSNE(n_components=2, random_state=42, perplexity=min(30, len(embeddings)-1))
    embeddings_2d = tsne.fit_transform(embeddings)
    
    # Plot
    fig, ax = plt.subplots(figsize=(10, 8))
    
    for i, name in enumerate(label_names):
        mask = labels == i
        ax.scatter(
            embeddings_2d[mask, 0],
            embeddings_2d[mask, 1],
            label=name,
            alpha=0.7,
            s=50
        )
    
    ax.set_xlabel('t-SNE 1')
    ax.set_ylabel('t-SNE 2')
    ax.set_title(title)
    ax.legend()
    plt.tight_layout()
    
    return fig

In [ ]:
# Extract embeddings from encoder
def extract_embeddings(encoder, sequences: list, max_len: int = 1500):
    """Extract embeddings from encoder."""
    encoder.eval()
    seq_encoder = SequenceEncoder(max_len=max_len)
    
    embeddings = []
    with torch.no_grad():
        for seq in sequences:
            encoded = seq_encoder(seq).unsqueeze(0)
            mask = (encoded != 0).long()
            emb = encoder(encoded, mask)
            embeddings.append(emb.squeeze().numpy())
    
    return np.array(embeddings)

## 5. Model Comparison

In [ ]:
# Compare different encoder architectures
def compare_models(results_dict: dict, metric: str = "accuracy"):
    """Compare multiple models across tasks."""
    tasks = list(list(results_dict.values())[0].keys())
    models = list(results_dict.keys())
    
    fig, ax = plt.subplots(figsize=(10, 6))
    
    x = np.arange(len(tasks))
    width = 0.8 / len(models)
    
    for i, model_name in enumerate(models):
        values = [results_dict[model_name][task][metric] for task in tasks]
        ax.bar(x + i * width, values, width, label=model_name)
    
    ax.set_xlabel('Task')
    ax.set_ylabel(metric.capitalize())
    ax.set_title(f'Model Comparison - {metric.capitalize()}')
    ax.set_xticks(x + width * (len(models) - 1) / 2)
    ax.set_xticklabels([t.capitalize() for t in tasks])
    ax.legend()
    ax.set_ylim(0, 1)
    
    plt.tight_layout()
    return fig

## 6. Export Results

In [ ]:
# Save predictions to CSV
def export_predictions(predictions: list, output_path: Path):
    """Export predictions to CSV file."""
    rows = []
    for i, pred in enumerate(predictions):
        row = {"sample_id": i}
        for task, value in pred["predictions"].items():
            row[f"{task}_pred"] = value
        for task, probs in pred["probabilities"].items():
            for j, p in enumerate(probs):
                row[f"{task}_prob_{j}"] = p
        rows.append(row)
    
    df = pd.DataFrame(rows)
    df.to_csv(output_path, index=False)
    print(f"Exported predictions to {output_path}")
    return df

In [ ]:
print("Analysis notebook ready!")
print("\nNext steps:")
print("1. Place your data in the 'data' directory")
print("2. Train a model using 'python scripts/run.py'")
print("3. Load the checkpoint and evaluate")